# Quick start

In [ ]:
# Setup. On Google Colab this downloads the course files and installs the dependencies (~2 min).
# Locally (conda env from environment.yml) it only moves to the course folder so `data/` is found.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/cell-factory-design-course"):
        !git clone -q --depth 1 https://github.com/lab-biotek-bio-ugm/cell-factory-design-course /content/cell-factory-design-course
        %pip install -q --progress-bar off -r /content/cell-factory-design-course/requirements.txt
    from google.colab import output
    output.enable_custom_widget_manager()  # needed for Escher maps

for folder in (".", "..", "/content/cell-factory-design-course"):
    if os.path.isdir(os.path.join(folder, "data")):
        os.chdir(folder)
        break

## Overview

**Learning objectives.** After this notebook you can:
1. Load a genome-scale metabolic model from an SBML file with cobrapy.
2. Run flux balance analysis (FBA) with `model.optimize()`.
3. Read the predicted growth rate and individual reaction fluxes from the solution.
4. Explain what the numbers mean (units, sign convention).

**Estimated time:** 30 min (15 min walkthrough, 15 min exercise).

**Prerequisites:** basic Python (variables, `for` loops, attribute access with `.`), first-year biochemistry (glycolysis, TCA cycle).

### Background

A **genome-scale metabolic model (GEM)** is a list of all metabolic reactions an organism is known to be able to carry out, reconstructed from its genome annotation and the biochemical literature. For every reaction the model stores its stoichiometry (which metabolites are consumed and produced, and in what ratio), its reversibility, and the genes that encode the enzyme(s) catalysing it. Here we use *i*JO1366, a model of *Escherichia coli* K-12 MG1655 with 1366 genes ([Orth et al., 2011](https://doi.org/10.1038/msb.2011.65)).

Models are exchanged as **SBML** (Systems Biology Markup Language) files, an XML format. Curated models for many organisms can be downloaded from the **BiGG Models** database ([bigg.ucsd.edu](https://bigg.ucsd.edu)), which also uses a consistent naming scheme for metabolites and reactions (e.g. `glc__D_e` is D-glucose in the extracellular space). The notebooks load and simulate models with **cobrapy** ([Ebrahim et al., 2013](https://doi.org/10.1186/1752-0509-7-74)).

**Flux balance analysis (FBA)** predicts the reaction rates (fluxes) $v$ of the whole network by assuming that the cell is in a (pseudo) steady state, so that internal metabolites neither accumulate nor deplete:

$$
\begin{aligned}
\max_{v} \quad & c^\top v \quad &\text{(objective, usually the biomass reaction = growth)}\\
\text{subject to} \quad & S\,v = 0 \quad &\text{(mass balance for every metabolite)}\\
& v_{\min} \le v \le v_{\max} \quad &\text{(flux bounds: reversibility, nutrient uptake limits)}
\end{aligned}
$$

$S$ is the **stoichiometric matrix** (rows = metabolites, columns = reactions). This is a **linear program** and is solved in milliseconds even for thousands of reactions. Fluxes are expressed in **mmol per gram dry weight per hour (mmol gDW⁻¹ h⁻¹)**; the flux of the biomass reaction is the specific growth rate in **h⁻¹**. By convention, a negative flux through an exchange reaction (e.g. `EX_glc__D_e`) means uptake, a positive one means secretion.

FBA does not need enzyme kinetics, which is why it scales to whole genomes. The price is that it only predicts what is *possible* given the constraints and an assumed objective, not what the cell actually regulates itself to do ([Orth et al., 2010](https://doi.org/10.1038/nbt.1614)).

## Loading a model

`read_sbml_model` parses the (gzip-compressed) SBML file into a `cobra.Model` object that holds metabolites, reactions and genes.

In [ ]:
from cobra.io import read_sbml_model

In [ ]:
model = read_sbml_model('data/iJO1366.xml.gz')

## Simulations

`model.optimize()` builds the linear program described above and hands it to a solver (here GLPK, which is installed with cobrapy). By default the objective is the model's biomass reaction, so FBA maximizes the growth rate given the default medium (glucose minimal medium with oxygen).

In [ ]:
solution = model.optimize()

## Reaction fluxes

After optimizing, every reaction carries a flux. Most of the ~2600 reactions carry zero flux: they are not needed for optimal growth on glucose.

In [ ]:
for reaction in model.reactions:
    print(reaction.id, reaction.flux, reaction.name)

The growth rate is 0.98 $h^{-1}$, which corresponds to a doubling time of $\ln 2 / 0.98 \approx 0.7$ h (about 42 min). This is close to what *E. coli* achieves in glucose minimal medium, but remember that it is a *maximum* allowed by the constraints: it depends directly on the glucose uptake bound (10 mmol gDW⁻¹ h⁻¹ here).

In [ ]:
model.reactions.BIOMASS_Ec_iJO1366_core_53p95M.flux

`solution.to_frame()` returns a pandas DataFrame with two columns: `fluxes` and `reduced_costs`. The **reduced cost** of a reaction tells you how much the objective would change per unit of flux if that reaction were forced to carry flux; it is zero for reactions that are already used in the optimum. Filtering for non-zero fluxes shows the active part of the network.

In [ ]:
solution_frame = solution.to_frame()
solution_frame[solution_frame.fluxes != 0.]

In [ ]:
solution_frame

## Exercise

1. Go to [bigg.ucsd.edu](https://bigg.ucsd.edu) and download a model in SBML format.
1. Load the model using `cobra.io.read_sbml_model` (or `cobra.io.load_model("<BiGG id>")`, which downloads it for you).
1. Optimize the flux distribution using regular flux balance analysis.
1. What is the objective value?

### Solution

We download *E. coli* core (`e_coli_core`) from BiGG. If there is no internet connection, we fall back to the copy bundled in `data/`.

In [ ]:
import os
import tempfile
import urllib.request
from cobra.io import read_sbml_model

try:
    path = os.path.join(tempfile.gettempdir(), "e_coli_core_bigg.xml.gz")
    urllib.request.urlretrieve("https://bigg.ucsd.edu/static/models/e_coli_core.xml.gz", path)
    my_model = read_sbml_model(path)
except Exception as error:  # offline: use the bundled copy
    print("Download failed, using data/e_coli_core.xml.gz:", error)
    my_model = read_sbml_model("data/e_coli_core.xml.gz")

my_solution = my_model.optimize()
my_solution.objective_value

The objective value is the maximal growth rate, about 0.87 h⁻¹ for the core model on 10 mmol gDW⁻¹ h⁻¹ glucose. It is lower than for *i*JO1366 mainly because the two models use different biomass reactions and growth/maintenance ATP requirements.

## References

- Ebrahim, Lerman, Palsson et al. (2013). COBRApy: COnstraints-Based Reconstruction and Analysis for Python. *BMC Systems Biology* 7:74. [doi.org/10.1186/1752-0509-7-74](https://doi.org/10.1186/1752-0509-7-74)
- Orth, Thiele, Palsson (2010). What is flux balance analysis? *Nature Biotechnology* 28:245–248. [doi.org/10.1038/nbt.1614](https://doi.org/10.1038/nbt.1614)
- Orth, Conrad, Na et al. (2011). A comprehensive genome‐scale reconstruction of Escherichia coli metabolism—2011. *Molecular Systems Biology* 7:535. [doi.org/10.1038/msb.2011.65](https://doi.org/10.1038/msb.2011.65)